##### Copyright 2025 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Handling Errors and Retries with the Gemini API

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Handling_Errors_and_Retries.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

When building production applications, API requests can occasionally fail due to rate limits (HTTP `429`) or temporary server-side overloads (HTTP `500`, `503`). 

To build resilient applications, you should configure **retry logic with exponential backoff**. This technique catches these specific errors and automatically retries the request, pausing for progressively longer intervals between each attempt (for example: wait 2 seconds, then 4 seconds, then 8 seconds).

This quickstart demonstrates how to use the built-in retry features in the `google-genai` SDK.

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [ ]:
!pip install -U google-genai

### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](./Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

### Choose a model

Select the model you want to use in this guide. You can either select one from the list or enter a model name manually. Keep in mind that some models, such as the 2.5 ones are thinking models and thus take slightly more time to respond. For more details, you can see [thinking notebook ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Get_started_thinking.ipynb) to learn how to control the thinking.

Feel free to select [Gemini 3 Pro](https://ai.google.dev/gemini-api/docs/models#gemini-3-pro) if you want to try our newest model, but keep in mind that it has no free tier.

For a full overview of all Gemini models, check the [documentation](https://ai.google.dev/gemini-api/docs/models/gemini).

In [ ]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, "isTemplate": true}

## Configure HttpRetryOptions

The `google-genai` SDK natively supports automatic retries through the `HttpRetryOptions` configuration. You can pass these options when initializing the client to automatically handle transient errors across all your API calls.

* `attempts`: The maximum number of times to retry the request.
* `initial_delay`: The amount of time (in seconds) to wait before the first retry.
* `max_delay`: The absolute maximum amount of time (in seconds) to wait between retries.
* `http_status_codes`: A list of HTTP status codes that should trigger a retry. Standard practice is to retry on `429` (Rate Limited) and `5xx` (Server Errors).

In [ ]:
client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            attempts=5,
            initial_delay=2.0,
            max_delay=60.0,
            http_status_codes=[429, 500, 502, 503, 504]
        )
    )
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents='Explain the importance of exponential backoff when calling APIs in one short paragraph.'
)

print(response.text)